# Stupid tests notebook
A playground notebook where I mostly compare some dummy statistics or do some quick checks.
I'll commit as it is, but it's in NO WAY running fully, and should not be use as an example.

@TODO: Eventually delete that notebook when releasing the repo

In [ ]:
import sys
import numpy as np
from pathlib import Path
import rootutils
import os

rootutils.setup_root("./stupid_tests.ipynb", indicator=".project-root", pythonpath=True)
# print env PROJECT_ROOT
print("PROJECT_ROOT:", os.getenv("PROJECT_ROOT"))

from src.utils.constants import amp_max, amp_min  # noqa E402
from src.utils.metrics import get_all_distortion_metrics  # noqa E402


def print_statistics(data, name):
    print(
        f"{name:<40}: min={np.min(data):.6f}, max={np.max(data):.6f}, mean={np.mean(data):.6f}, std={np.std(data):.6f}"
    )


def print_metrics(name, pred, gt, max=None):
    metrics = get_all_distortion_metrics(pred, gt)
    if max is None:
        max = np.max(gt) - np.min(gt)
    print(f"{name:<40} (max={max:.6f}):")
    print(
        f"  - MSE={metrics['mse']:.6f}, PSNR={metrics['psnr']:.6f}dB, SSIM={metrics['ssim']:.6f}, MS-SSIM={metrics['ms_ssim']:.6f}"
    )
    print(
        f"  - pred: min={np.min(pred):.6f}, max={np.max(pred):.6f}, mean={np.mean(pred):.6f}, std={np.std(pred):.6f}"
    )
    print(
        f"  - gt  : min={np.min(gt):.6f}, max={np.max(gt):.6f}, mean={np.mean(gt):.6f}, std={np.std(gt):.6f}"
    )


# Initialize a random 256x256x2 array with values 0.767725, max=1.480479, mean=0.985091
data = np.random.rand(256, 256, 2) * (1.480479 - 0.767725) + 0.767725
print_statistics(data, "normalized data")
# denormalize the data
data_logI = data * (amp_max - amp_min) + amp_min
print_statistics(data_logI, "denormalized data (logI)")
data_lin = np.exp(data_logI)
print_statistics(data_lin, "denormalized data (lin)")
data_lin_I = 0.5 * (np.square(data_lin[..., 0]) + np.square(data_lin[..., 1]))
data_lin_A_from_merged = np.sqrt(np.square(data_logI_merged))
print_statistics(data_lin_A_from_merged, "lin A from merged logI")
data_lin_A_merged = np.sqrt(0.5 * (np.square(data_lin[..., 0]) + np.square(data_lin[..., 1])))
print_statistics(data_lin_A_merged, "merged denormalized data (lin A)")
data_merged_back_to_log_I = np.log(np.sqrt(data_lin_A_merged))
print_statistics(data_merged_back_to_log_I, "merged back to logI")

PROJECT_ROOT: /mnt/vitisAI/Vitis-AI/DDC_FPGA
normalized data                         : min=0.767732, max=1.480475, mean=1.124274, std=0.206072
denormalized data (logI)                : min=9.316798, max=13.690948, mean=11.504921, std=1.264675
merged denormalized data (logI)         : min=9.340053, max=13.680220, mean=11.504921, std=0.893550
denormalized data (lin)                 : min=11123.303550, max=882882.782440, mean=199866.509201, std=222863.688577
lin A from merged logI                  : min=9.340053, max=13.680220, mean=11.504921, std=0.893550
merged denormalized data (lin A)        : min=11388.878103, max=873504.968838, mean=236065.778193, std=184086.375250
merged back to logI                     : min=4.670196, max=6.840135, mean=5.985518, std=0.497923


In [ ]:
import h5py
from omegaconf import OmegaConf, DictConfig
import hydra
from lightning import LightningModule
import torch

from src import data
from src.data.sar_datamodule import TSXSSCDataModule

from typing import Any


def _load_training_cfg_from_ckpt(ckpt_path: Path) -> Any:
    """Load the original training configuration from the checkpoint's run directory."""
    run_dir = ckpt_path.parent.parent
    training_config_path = run_dir / ".hydra" / "config.yaml"
    if not training_config_path.exists():
        raise FileNotFoundError(f"Training config not found at {training_config_path}.")
    print(f"Loading original training config from {training_config_path}")
    return OmegaConf.load(training_config_path)


def _instantiate_model_and_load_weights(train_cfg: DictConfig, ckpt_path: Path) -> LightningModule:
    """Instantiate the model from training config and load weights from checkpoint."""
    print(f"Instantiating model <{train_cfg.model._target_}>")
    model: LightningModule = hydra.utils.instantiate(train_cfg.model)
    checkpoint = torch.load(str(ckpt_path), map_location="cpu")
    msg = model.load_state_dict(checkpoint["state_dict"], strict=False)
    print(f"Loaded checkpoint state_dict with message: {msg}")
    model.eval()
    return model


batch_size = 10
EPS = 1e-2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ------------------------------------------------ LOAD MODEL ------------------------------------------------
# ckpt = Path("../logs/train/sar_ddc/hyperprior/runs/2025-09-18_13-05-43/checkpoints/last.ckpt")
ckpt = Path(
    "../logs/train/sar_ddc/hyperprior/multiruns/2025-11-23_10-18-18/0/checkpoints/last.ckpt"
)
train_cfg = _load_training_cfg_from_ckpt(ckpt)
model = _instantiate_model_and_load_weights(train_cfg, ckpt)
model.to(device)

Loading original training config from ../logs/train/sar_ddc/hyperprior/multiruns/2025-11-23_10-18-18/0/.hydra/config.yaml
Instantiating model <src.models.sar_ddc_module.SARDDCModule>
Loaded checkpoint state_dict with message: _IncompatibleKeys(missing_keys=['net.entropy_bottleneck.matrices.0', 'net.entropy_bottleneck.matrices.1', 'net.entropy_bottleneck.matrices.2', 'net.entropy_bottleneck.matrices.3', 'net.entropy_bottleneck.matrices.4', 'net.entropy_bottleneck.biases.0', 'net.entropy_bottleneck.biases.1', 'net.entropy_bottleneck.biases.2', 'net.entropy_bottleneck.biases.3', 'net.entropy_bottleneck.biases.4', 'net.entropy_bottleneck.factors.0', 'net.entropy_bottleneck.factors.1', 'net.entropy_bottleneck.factors.2', 'net.entropy_bottleneck.factors.3'], unexpected_keys=['net.entropy_bottleneck._matrix0', 'net.entropy_bottleneck._bias0', 'net.entropy_bottleneck._factor0', 'net.entropy_bottleneck._matrix1', 'net.entropy_bottleneck._bias1', 'net.entropy_bottleneck._factor1', 'net.entropy_b

/tmp/ipykernel_1094945/3861082779.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(str(ckpt_path), map_location="cpu")


SARDDCModule(
  (net): ResidualScaleHyperprior(
    (entropy_bottleneck): EntropyBottleneck(
      (likelihood_lower_bound): LowerBound()
      (matrices): ParameterList(
          (0): Parameter containing: [torch.float32 of size 256x3x1 (cuda:0)]
          (1): Parameter containing: [torch.float32 of size 256x3x3 (cuda:0)]
          (2): Parameter containing: [torch.float32 of size 256x3x3 (cuda:0)]
          (3): Parameter containing: [torch.float32 of size 256x3x3 (cuda:0)]
          (4): Parameter containing: [torch.float32 of size 256x1x3 (cuda:0)]
      )
      (biases): ParameterList(
          (0): Parameter containing: [torch.float32 of size 256x3x1 (cuda:0)]
          (1): Parameter containing: [torch.float32 of size 256x3x1 (cuda:0)]
          (2): Parameter containing: [torch.float32 of size 256x3x1 (cuda:0)]
          (3): Parameter containing: [torch.float32 of size 256x3x1 (cuda:0)]
          (4): Parameter containing: [torch.float32 of size 256x1x1 (cuda:0)]
      )
  

In [ ]:
# ------------------------------------------------ OPEN DATA ------------------------------------------------
with h5py.File(
    "../data/processed_hdf5/TSX_preprocessed_spatial_splits_5_256x256/test.h5", "r"
) as f:
    test_input_data = f["patches"][0:batch_size]
print(f"Input data shape: {test_input_data.shape}\n")
print_statistics(test_input_data, "Input Data symmetrized")

noisy_I = (np.square(test_input_data[:, :, :, 0]) + np.square(test_input_data[:, :, :, 1]))[
    :, np.newaxis, :, :
]
print_statistics(noisy_I, "Noisy Input Data lin")
noisy_logI = np.log(noisy_I + EPS)
print_statistics(noisy_logI, "Noisy Input Data logI")

input_norm = torch.from_numpy(test_input_data).float()
input_norm = (torch.log(torch.square(input_norm) + EPS) - 2 * amp_min) / (
    2 * amp_max - 2 * amp_min
)
input_norm = input_norm.permute(0, 3, 1, 2).float().to(device)
print_statistics(input_norm[:, :, :].cpu().numpy(), "Input Data normalized")
print()

with torch.inference_mode():
    out = model(input_norm)
    input_norm = input_norm.cpu().numpy()

recon = out["x_hat"].cpu().numpy()
print_statistics(recon, "Output as reconstructed")
recon_merged = 0.5 * (recon[:, :1, :, :] + recon[:, 1:, :, :])
print_statistics(recon_merged, "Output merged")

recon_lin = np.exp(recon * (amp_max - amp_min) + amp_min)
print_statistics(recon_lin, "Output as reconstructed denormed lin")
recon_lin = 0.5 * (recon_lin[:, :1, :, :] + recon_lin[:, 1:, :, :])
print_statistics(recon_lin, "Output merged denormed lin")
print()

print_metrics("Input Vs Output", recon, input_norm)
print_metrics("Input Vs Output (Max = 1)", recon, input_norm, max=1.0)

print_metrics(
    "Input Vs Output (merged)",
    recon_merged,
    0.5 * (input_norm[:, :1, :, :] + input_norm[:, 1:, :, :]),
)
print_metrics(
    "Input Vs Output (merged) (Max = DATA_RANGE)",
    recon_merged,
    0.5 * (input_norm[:, :1, :, :] + input_norm[:, 1:, :, :]),
    max=2 * amp_max - 2 * amp_min,
)
print_metrics("Input Vs Output lin (merged)", recon_lin, noisy_I)


# The PSNR must be computed on linear images. I need to change that. Probably amplitude images.
# The whole image manipulation is done as such:

print("\n\n------------- MANUAL TESTING OF DATA MANIPULATION -------------")
# `image` is directly read from COS files
image_sym = test_input_data  # shape (B, H, W, 2)
print_statistics(image_sym, "Original Input Data symmetrized")
image_sym_real = image_sym[:, :, :, 0]
image_sym_imaginary = image_sym[:, :, :, 1]
noisy_im = np.sqrt(np.square(image_sym_real) + np.square(image_sym_imaginary))
print_statistics(noisy_im, "Noisy Input Data lin amplitude")
# `image_norm` is computed to be the input of the model
# (torch.log(torch.square(image_real_part) + 1e-3) - 2 * m) / (2 * (M - m))
image_log = np.log(np.square(image_sym) + 0.01)
image_log_norm = (image_log - 2 * amp_min) / (2 * amp_max - 2 * amp_min)
print_statistics(image_log_norm, "Input Data log normalized")
# `recon` is generated by the model (who is fed with `image_log_norm`)
print_statistics(recon, "Reconstructed Data")
# recon = model(image_log_norm)
recon_denorm = recon * (amp_max - amp_min) + amp_min
print_statistics(recon_denorm, "Reconstructed Data denormalized")
recon_denorm_lin = np.exp(recon_denorm)
print_statistics(recon_denorm_lin, "Reconstructed Data denormalized lin")
# `clean_im` is the average of the denormalized reconstruction
clean_im_real = np.square(recon_denorm_lin[:, 0, :, :])
clean_im_imag = np.square(recon_denorm_lin[:, 1, :, :])
clean_im = np.sqrt(0.5 * (clean_im_real + clean_im_imag))
print_statistics(clean_im, "Clean Image lin amplitude")


# `PSNR` is computed between noisy and clean_image
def psnr_new(clean, noisy):
    mse = np.mean(np.square(clean - noisy))
    max = np.max(clean)
    return 20 * np.log10(max) - 10 * np.log10(mse)


psnr = psnr_new(clean_im, noisy_im)
print_metrics("Clean Vs Noisy images linear amplitude", clean_im, noisy_im)
print(f"Manually computed PSNR: {psnr:.2f} dB")

# How we manipulate the data in this project
# # ----- Normalization -----
# input_norm = torch.from_numpy(input_data).float()
# input_norm = (torch.log(torch.square(input_norm) + EPS) - 2 * amp_min) / (
#     2 * amp_max - 2 * amp_min
# )
# # ----- Forward -----
# recon = model(input_norm)
# # ----- PSNR -----
# # must be done on linear amplitude image
# noisy = 0.5 * (input_norm[:, :1, :, :] + input_norm[:, 1:, :, :])
# recon_merged = 0.5 * (recon[:, :1, :, :] + recon[:, 1:, :, :])


# def psnr_merlinsar(img1, img2):
#     # Output - noisy
#     mse = np.mean((img1 - img2) ** 2)
#     max_value = np.quantile(img1, 0.99)
#     return 10 * np.log10((max_value**2) / mse)

# print(psnr_merlinsar(recon_merged.numpy(), noisy.numpy()))


# output_data_denorm_merlin_sar = np.exp(
#     (np.clip(np.squeeze(output_data), 0, 1)) * (amp_max - amp_min) + amp_min
# )

Input data shape: (10, 256, 256, 2)

Input Data symmetrized                  : min=-7855.974121, max=16855.757812, mean=0.099955, std=128.793823
Noisy Input Data lin                    : min=0.000000, max=297551968.000000, mean=33175.707031, std=530064.250000
Noisy Input Data logI                   : min=-4.605170, max=19.511099, mean=9.275851, std=1.571434
Input Data normalized                   : min=0.150031, max=0.934204, mean=0.557198, std=0.077688

Output as reconstructed                 : min=0.767725, max=1.480479, mean=0.985091, std=0.055367
Output merged                           : min=0.773653, max=1.383849, mean=0.985092, std=0.053682
Output as reconstructed denormed lin    : min=1309.516113, max=73761840.000000, mean=70526.375000, std=547323.812500
Output merged denormed lin              : min=1434.799561, max=37732892.000000, mean=70526.359375, std=424767.218750

Input Vs Output                          (max=0.784174):
  - MSE=0.189992, PSNR=5.100883dB
  - pred: min=0.767

## Merlin SAR code
See https://github.com/pbla749/merlin-sar/blob/main/merlinsar/train/model.py#L151

In [ ]:
M = amp_max
m = amp_min


def denormalize_sar(im):
    return np.exp((np.clip(np.squeeze(im), 0, 1)) * (M - m) + m)


def cal_psnr(Shat, S):
    # takes amplitudes in input
    # Shat: a SAR amplitude image
    # S:    a reference SAR image
    P = np.quantile(S, 0.99)
    res = 10 * np.log10((P**2) / np.mean(np.abs(Shat - S) ** 2))
    return res


image_real_part = torch.from_numpy(test_input_data[:, :, :, 0])
image_imaginary_part = torch.from_numpy(test_input_data[:, :, :, 1])

image_real_part = image_real_part.to(device)
print_statistics(image_real_part.cpu().numpy(), "Real Part")
image_imaginary_part = image_imaginary_part.to(device)

# Normalization
image_real_part_normalized = (torch.log(torch.square(image_real_part) + 1e-3) - 2 * m) / (
    2 * (M - m)
)
print_statistics(image_real_part_normalized.cpu().numpy(), "Real Part Normalized")
image_imaginary_part_normalized = (
    torch.log(torch.square(image_imaginary_part) + 1e-3) - 2 * m
) / (2 * (M - m))

out_real = out["x_hat"][:, 0, :, :]  # forward(image_real_part_normalized, eval_batch_size)
print_statistics(out_real.cpu().numpy(), "Real Part Output")
out_imaginary = out["x_hat"][
    :, 1, :, :
]  # forward(image_imaginary_part_normalized, eval_batch_size)

output_clean_image = 0.5 * (
    np.square(denormalize_sar(out_real.cpu().numpy()))
    + np.square(denormalize_sar(out_imaginary.cpu().numpy()))
)
print_statistics(output_clean_image, "Output Clean Image")

# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # #

noisyimage = np.squeeze(
    np.sqrt(
        np.square(image_real_part.cpu().numpy()) + np.square(image_imaginary_part.cpu().numpy())
    )
)
print_statistics(noisyimage, "Noisy Image")
outputimage = np.sqrt(np.squeeze(output_clean_image))
print_statistics(outputimage, "Output Image")

# calculate PSNR
psnr = cal_psnr(outputimage, noisyimage)
print("PSNR: %.2f" % (psnr))

print_metrics("outputimage Vs noisyimage", outputimage, noisyimage)

Real Part                               : min=-7461.102051, max=7284.560547, mean=-0.111876, std=126.561378
Real Part Normalized                    : min=0.075015, max=0.881102, mean=0.557091, std=0.078007
Real Part Output                        : min=0.767725, max=1.480479, mean=0.985298, std=0.055841
Output Clean Image                      : min=2060248.750000, max=2140873856.000000, mean=1237539072.000000, std=720232640.000000
Noisy Image                             : min=0.000009, max=17249.695312, mean=135.946625, std=121.219765
Output Image                            : min=1435.356689, max=46269.578125, mean=33176.699219, std=11698.107422
PSNR: -36.83
outputimage Vs noisyimage                (max=17249.695312):
  - MSE=1227645312.000000, PSNR=-6.155101dB
  - pred: min=1435.356689, max=46269.578125, mean=33176.699219, std=11698.107422
  - gt  : min=0.000009, max=17249.695312, mean=135.946625, std=121.219765
